# 04 — LangGraph Sessions with MemorySaver

LangGraph has **built-in memory** via a *checkpointer*.
Every time the graph runs, the full state is saved automatically.
The next run picks up exactly where it left off.

### Key idea: `thread_id`
A `thread_id` is like a **session ID**. Every user (or conversation) gets their own.
```
thread "alice" → [Hi I'm Alice] → [What's my name?] → remembers "Alice"
thread "bob"   → [Hi I'm Bob]   → [What's my name?] → remembers "Bob"
```
The two threads are **completely independent**, even though they share the same graph.

### What you will learn
| Concept | Description |
|---------|-------------|
| `MemorySaver` | In-memory checkpointer — saves state after every node |
| `thread_id` | Unique key that identifies a conversation session |
| Multi-user sessions | Multiple threads running in parallel on the same graph |
| State inspection | Read back the saved state to see what was remembered |

> **Requires:** `GROQ_API_KEY` in a `.env` file

## 1. Install & Import

In [ ]:
# !pip install langgraph langchain-groq python-dotenv

In [ ]:
import os
from typing import Annotated
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from typing_extensions import TypedDict

load_dotenv()
print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))

## 2. Define the Graph State

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]
    # The 'add_messages' reducer APPENDS new messages
    # instead of replacing the entire list.
    # This is what gives us automatic history.

print("State defined!")
print("  messages: Annotated[list, add_messages]")
print("  → new messages are appended, never overwritten")

## 3. Build the Chatbot Node

In [ ]:
llm = ChatGroq(model="llama3-8b-8192", temperature=0.7)

SYSTEM = SystemMessage(content=(
    "You are a friendly assistant with a great memory. "
    "Remember everything the user tells you about themselves. "
    "Use their name and personal details when replying."
))

def chatbot_node(state: ChatState) -> dict:
    """The only node — call the LLM with the full message history."""    # Prepend the system message to the stored history
    messages = [SYSTEM] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}    # add_messages will append this

print("Chatbot node defined!")

## 4. Compile the Graph WITH a Checkpointer

In [ ]:
# ── Build the graph ───────────────────────────────────────────────────
builder = StateGraph(ChatState)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# ── Attach the MemorySaver ────────────────────────────────────────────
#    Without this, every invoke() starts fresh.
#    With this, state is saved after every run.
checkpointer = MemorySaver()
graph        = builder.compile(checkpointer=checkpointer)

print("Graph compiled with MemorySaver!")
print("Every run will now save + restore state automatically.")

## 5. The `config` Dictionary

Every `invoke()` call needs a `config` that specifies which thread to use.

```python
config = {"configurable": {"thread_id": "alice-001"}}
```

Think of `thread_id` as a **drawer label** — the checkpointer stores each
conversation in its own labelled drawer and pulls out the right one each time.

In [ ]:
def send_message(text: str, thread_id: str) -> str:
    """Send a message in a specific conversation thread."""    config   = {"configurable": {"thread_id": thread_id}}
    result   = graph.invoke({"messages": [HumanMessage(content=text)]}, config)
    reply    = result["messages"][-1].content
    total    = len(result["messages"])
    print(f"  [{thread_id}] You: {text}")
    print(f"  [{thread_id}] Bot: {reply}")
    print(f"  [{thread_id}] (Total messages in session: {total})\n")
    return reply

print("send_message() helper ready!")

## 6. Two Users — Two Separate Sessions

In [ ]:
# ── Alice's session ───────────────────────────────────────────────────
print("=" * 50)
print("ALICE'S CONVERSATION")
print("=" * 50 + "\n")

send_message("Hi! My name is Alice. I'm a doctor from Delhi.", "alice")
send_message("I have a 7-year-old daughter named Zara.",       "alice")
send_message("What is my name?",                               "alice")

In [ ]:
# ── Bob's session — completely separate ───────────────────────────────
print("=" * 50)
print("BOB'S CONVERSATION")
print("=" * 50 + "\n")

send_message("Hey, I'm Bob. Software engineer from Chennai.", "bob")
send_message("I'm learning LangGraph this week.",            "bob")
send_message("What's my name and what am I learning?",       "bob")

In [ ]:
# ── Back to Alice — the graph remembers her ────────────────────────────
print("=" * 50)
print("BACK TO ALICE (new invoke, same thread)")
print("=" * 50 + "\n")

send_message("What do you know about my family?", "alice")
send_message("And what city am I from?",          "alice")

## 7. Inspect the Saved State

In [ ]:
# You can read back the full saved state at any time
def show_session_state(thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}
    state  = graph.get_state(config)

    print(f"\n--- State snapshot for thread: '{thread_id}' ---")
    messages = state.values.get("messages", [])
    print(f"Total messages saved: {len(messages)}")
    for i, msg in enumerate(messages):
        role    = "Human" if isinstance(msg, HumanMessage) else "AI"
        preview = msg.content[:70] + "..." if len(msg.content) > 70 else msg.content
        print(f"  [{i+1:2d}] {role:5s}: {preview}")
    print()

show_session_state("alice")
show_session_state("bob")

## 8. What Happens With an Unknown Thread?

In [ ]:
# A brand-new thread starts with no history
print("Sending a message as a completely new user (thread: 'charlie'):\n")
send_message("What's my name?", "charlie")
# The bot doesn't know — this thread has no history yet

## 9. Key Takeaways

| Concept | What you learned |
|---------|-----------------|
| `MemorySaver` | Checkpointer that saves graph state after every run |
| `thread_id` | Unique session key — each user gets their own thread |
| `add_messages` | Reducer that appends messages instead of replacing them |
| `graph.get_state(config)` | Read back the saved state for any thread |
| Multi-user | Many threads can run on the same compiled graph |

> **Limitation:** `MemorySaver` stores data **in RAM**.
> If the Python process restarts, all sessions are lost.
> Notebook 05 shows how to save memory to a file so it survives restarts.

**Next notebook →** save memory to a JSON file for true long-term memory.